# Giải quyết bài toán 8-Puzzle bằng thuật toán Leo Núi (Hill Climbing)


In [51]:
import random
import time

## 1. Khởi tạo và Tiện ích


In [52]:
GOAL = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
OPPOSITES = {'UP': 'DOWN', 'DOWN': 'UP', 'LEFT': 'RIGHT', 'RIGHT': 'LEFT'}

def is_solvable(state):
    """Đếm số inversions để kiểm tra xem trạng thái có thể giải được không."""
    flat = [val for row in state for val in row if val != 0]
    inversions = sum(
        1 for i in range(len(flat)) for j in range(i + 1, len(flat))
        if flat[i] > flat[j]
    )
    return inversions % 2 == 0

def generate_random_board():
    """Khởi tạo bàn cờ ngẫu nhiên hợp lệ (luôn giải được)."""
    while True:
        nums = list(range(9))
        random.shuffle(nums)
        board = [nums[i:i + 3] for i in range(0, 9, 3)]
        if is_solvable(board):
            return board

def generate_near_goal_board(k):
    """Sinh bàn cờ từ trạng thái đích bằng cách đi ngẫu nhiên k bước hợp lệ."""
    board = [row[:] for row in GOAL]
    last_move = None
    for _ in range(k):
        moves = list(get_legal_moves(board).keys())
        if last_move and OPPOSITES[last_move] in moves:
            moves.remove(OPPOSITES[last_move])
        if not moves:
            break
        move = random.choice(moves)
        board = apply_move(board, move)
        last_move = move
    return board

def find_blank(board):
    """Tìm vị trí ô trống (0)."""
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c
    return None

def get_legal_moves(board):
    """Trả về các nước đi hợp lệ dưới dạng dict {hướng: (hàng_mới, cột_mới)}."""
    res = find_blank(board)
    if res is None:
        return {}
    r, c = res
    moves = {}
    if r > 0: moves['UP'] = (r - 1, c)
    if r < 2: moves['DOWN'] = (r + 1, c)
    if c > 0: moves['LEFT'] = (r, c - 1)
    if c < 2: moves['RIGHT'] = (r, c + 1)
    return moves

def apply_move(board, move):
    """Di chuyển ô trống và trả về bàn cờ mới (không thay đổi bàn cờ cũ)."""
    res = find_blank(board)
    if res is None:
        return board
    r, c = res
    nr, nc = get_legal_moves(board)[move]
    new_board = [row[:] for row in board]
    new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
    return new_board

def print_board(board, title=None):
    """In bàn cờ dạng lưới 3x3 dễ quan sát."""
    if title:
        print(f"\n{title}")
    for row in board:
        print(" ".join(f"[{n if n != 0 else ' '}]" for n in row))

## 2. Hàm Heuristic và Lượng Giá


In [53]:
GOAL_POSITIONS = {
    GOAL[r][c]: (r, c)
    for r in range(3) for c in range(3)
    if GOAL[r][c] != 0
}

def misplaced_tiles(board):
    """Heuristic đếm số ô sai vị trí (bỏ qua ô trống)."""
    count = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0 and val != GOAL[r][c]:
                count += 1
    return count

def manhattan_distance(board):
    """Heuristic khoảng cách Manhattan (bỏ qua ô trống)."""
    dist = 0
    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                gr, gc = GOAL_POSITIONS[val]
                dist += abs(r - gr) + abs(c - gc)
    return dist

# Định nghĩa hàm Value tương ứng (Leo núi tìm cực đại nên lấy âm của Heuristic)
def value_manhattan(board):
    return -manhattan_distance(board)

## 3. Thuật toán Leo núi Đơn giản (Simple Hill Climbing - Ảnh 1)


In [54]:
def Simple_Hill_Climbing(Start, value_func):
    """
    Thuật toán Leo núi Đơn giản (Simple Hill Climbing) theo Slide 1.
    """
    Current_State = Start
    path = [Current_State]
    moves = []
    
    while True:
        if Current_State == GOAL:
            return Current_State, path, moves, "SUCCESS"
            
        legal_moves = get_legal_moves(Current_State)
        found_better = False
        
        for move in legal_moves:
            Next_State = apply_move(Current_State, move)
            
            if value_func(Next_State) > value_func(Current_State):
                Current_State = Next_State
                path.append(Current_State)
                moves.append(move)
                found_better = True
                break  # Quay lại đầu vòng lặp
                
        if not found_better:
            return Current_State, path, moves, "LOCAL_MAXIMUM"

## 4. Thuật toán Leo núi Dốc nhất (Steepest Ascent Hill Climbing - Ảnh 2)


In [55]:
def Steepest_Ascent_Hill_Climbing(Start, value_func):
    """
    Thuật toán Leo núi Dốc nhất (Steepest Ascent Hill Climbing) theo Slide 2.
    """
    Current_State = Start
    path = [Current_State]
    moves = []
    
    while True:
        if Current_State == GOAL:
            return Current_State, path, moves, "SUCCESS"
            
        legal_moves = get_legal_moves(Current_State)
        if not legal_moves:
            return Current_State, path, moves, "LOCAL_MAXIMUM"
            
        neighbors = []
        for move in legal_moves:
            neighbor_state = apply_move(Current_State, move)
            neighbors.append((neighbor_state, move))
            
        best_neighbor, best_move = max(neighbors, key=lambda x: value_func(x[0]))
        
        if value_func(best_neighbor) > value_func(Current_State):
            Current_State = best_neighbor
            path.append(Current_State)
            moves.append(best_move)
        else:
            return Current_State, path, moves, "LOCAL_MAXIMUM"

## 5. Thử nghiệm với Bàn cờ Ngẫu nhiên


In [56]:
def run_and_print_steps(algorithm_func, name, start_board, value_func, heuristic_func):
    print("=" * 65)
    print(f" THUẬT TOÁN: {name} ")
    print("=" * 65)
    print_board(start_board, "TRẠNG THÁI BẮT ĐẦU:")
    print(f"Heuristic ban đầu = {heuristic_func(start_board)}")
    print("-" * 65)
    
    t0 = time.time()
    res_board, path, moves, status = algorithm_func(start_board, value_func)
    duration = time.time() - t0
    
    # Tái hiện từng bước di chuyển của bàn cờ
    current = start_board
    for step, move in enumerate(moves, 1):
        current = apply_move(current, move)
        print_board(current, f"=> Bước {step}: Đi [{move}] | Heuristic = {heuristic_func(current)}")
        
    print("-" * 65)
    print(f"KẾT QUẢ TỔNG QUAN:")
    print(f"  Kết quả cuối cùng      : {status}")
    print(f"  Tổng số bước di chuyển : {len(moves)}")
    print(f"  Thời gian thực thi     : {duration:.6f} giây")
    print(f"  Đường đi chi tiết      : {' -> '.join(moves) if moves else 'Không di chuyển'}")
    print("=" * 65)
    print("\n")

# Khởi tạo bàn cờ ngẫu nhiên (sinh bằng cách đi 8 bước ngẫu nhiên từ đích)
# Điều này đảm bảo có sự thay đổi và hiển thị các bước leo núi sinh động hơn
start_board = generate_near_goal_board(8)

# 1. Chạy thử nghiệm thuật toán Simple Hill Climbing
run_and_print_steps(Simple_Hill_Climbing, "Simple Hill Climbing", start_board, value_manhattan, manhattan_distance)

# 2. Chạy thử nghiệm thuật toán Steepest Ascent Hill Climbing
run_and_print_steps(Steepest_Ascent_Hill_Climbing, "Steepest Ascent Hill Climbing", start_board, value_manhattan, manhattan_distance)

 THUẬT TOÁN: Simple Hill Climbing 

TRẠNG THÁI BẮT ĐẦU:
[1] [2] [3]
[7] [6] [8]
[5] [4] [ ]
Heuristic ban đầu = 8
-----------------------------------------------------------------

=> Bước 1: Đi [UP] | Heuristic = 7
[1] [2] [3]
[7] [6] [ ]
[5] [4] [8]

=> Bước 2: Đi [LEFT] | Heuristic = 6
[1] [2] [3]
[7] [ ] [6]
[5] [4] [8]

=> Bước 3: Đi [DOWN] | Heuristic = 5
[1] [2] [3]
[7] [4] [6]
[5] [ ] [8]

=> Bước 4: Đi [LEFT] | Heuristic = 4
[1] [2] [3]
[7] [4] [6]
[ ] [5] [8]

=> Bước 5: Đi [UP] | Heuristic = 3
[1] [2] [3]
[ ] [4] [6]
[7] [5] [8]

=> Bước 6: Đi [RIGHT] | Heuristic = 2
[1] [2] [3]
[4] [ ] [6]
[7] [5] [8]

=> Bước 7: Đi [DOWN] | Heuristic = 1
[1] [2] [3]
[4] [5] [6]
[7] [ ] [8]

=> Bước 8: Đi [RIGHT] | Heuristic = 0
[1] [2] [3]
[4] [5] [6]
[7] [8] [ ]
-----------------------------------------------------------------
KẾT QUẢ TỔNG QUAN:
  Kết quả cuối cùng      : SUCCESS
  Tổng số bước di chuyển : 8
  Thời gian thực thi     : 0.000086 giây
  Đường đi chi tiết      : UP -> LEFT ->